In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
%matplotlib inline
# Retina 
%config InlineBackend.figure_format = 'retina'

In [ ]:
N = 500
D = 2

torch.manual_seed(1)
X = torch.distributions.MultivariateNormal(torch.zeros(D), torch.tensor([[2.0, 0.5], [0.5, 0.2]])).sample((N,))

X = X - X.mean(dim=0)
plt.scatter(X[:,0], X[:,1])
plt.gca().set_aspect('equal', adjustable='box')

In [ ]:
covariance_matrix = torch.matmul(X.t(), X) / (X.size(0)-1)
covariance_matrix.shape

In [ ]:
covariance_matrix

In [ ]:
eigenvalues, eigenvectors = torch.linalg.eig(covariance_matrix)

In [ ]:
eigenvalues = eigenvalues.real
eigenvalues

In [ ]:
eigenvectors = eigenvectors.real
eigenvectors

In [ ]:
# Plot the eigenvectors along with length of eigenvalues
plt.scatter(X[:,0], X[:,1])
plt.gca().set_aspect('equal', adjustable='box')
plt.quiver(0, 0, eigenvectors[0, 0], eigenvectors[1, 0], scale=eigenvalues[0].sqrt().item(), color='red', label='Eigenvector 1')
plt.quiver(0, 0, eigenvectors[0, 1], eigenvectors[1, 1], scale=eigenvalues[1].sqrt().item(), color='green', label='Eigenvector 2')
plt.legend()


In [ ]:
k = 1
Z = torch.matmul(X, eigenvectors[:, :k])
Z.shape

In [ ]:
plt.plot(Z, torch.zeros_like(Z), 'o')

In [ ]:
# Step 4: Sort eigenvectors based on eigenvalues
sorted_indices = torch.argsort(eigenvalues[:, 0], descending=True)
sorted_eigenvectors = eigenvectors[:, sorted_indices]


In [ ]:
sorted

In [ ]:
X.mean(dim=0), X.var(dim=0)

In [ ]:
k = 1  # Number of principal components
B = nn.Parameter(torch.randn(D, k))

In [ ]:
B.shape, X.shape

In [ ]:
mse_loss = nn.MSELoss()

optimizer = optim.Adam([B], lr=0.001) 
num_epochs = 2000  # Number of optimization epochs

for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Project data onto the current projection matrix
    z = torch.matmul(X, B)
    x_reconstructed = torch.matmul(z, B.t())
    
    # Compute reconstruction loss
    loss = mse_loss(x_reconstructed, X)
    
    # Perform backpropagation and optimization step
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item()}')


In [ ]:
# Project data onto the learned projection matrix
Z = torch.matmul(X, B)

In [ ]:
B

In [ ]:
def train_linear_autoencoder(X, K=2, lr=0.001, num_epochs=2000):
    D = X.size(1)
    B = nn.Parameter(torch.randn(D, K))
    
    mse_loss = nn.MSELoss()
    optimizer = optim.Adam([B], lr=lr)
    num_epochs = num_epochs
    
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        
        z = torch.matmul(X, B)
        x_reconstructed = torch.sigmoid(torch.matmul(z, B.t()))
        
        loss = mse_loss(x_reconstructed, X)
        
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item()}')
    
    return B

In [ ]:
# Get MNIST data
from torchvision import datasets
from torchvision.transforms import ToTensor

mnist_train = datasets.MNIST(root='~/.datasets', train=True, transform=ToTensor(), download=True)

In [ ]:
X = mnist_train.data.float()
X = X.view(X.size(0), -1)
X = X / 255.0

X.shape

In [ ]:
# Work with first 1000 samples
X = X[:3000]

In [ ]:
X.min(), X.max()

In [ ]:
# Plot the first 10 images
fig, ax = plt.subplots(1, 10, figsize=(10, 1))
for i in range(10):
    ax[i].imshow(X[i].view(28, 28).numpy(), cmap='gray')
    ax[i].axis('off')


In [ ]:
B = train_linear_autoencoder(X, K=5, lr=0.001, num_epochs=2000)

In [ ]:
B.shape

In [ ]:
# Project data onto the 2d space
with torch.no_grad():
    Z = torch.matmul(X, B)
    X_reconstructed = torch.sigmoid(torch.matmul(Z, B.t()))



In [ ]:
# Plot the first 10 images and their reconstructions
fig, ax = plt.subplots(2, 10, figsize=(10, 2))
for i in range(10):
    ax[0, i].imshow(X[i].view(28, 28).numpy(), cmap='gray')
    ax[0, i].axis('off')
    ax[1, i].imshow(X_reconstructed[i].view(28, 28).numpy(), cmap='gray')
    ax[1, i].axis('off')


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=32)

Z_pca = pca.fit_transform(X.numpy())

X_reconstructed_pca = pca.inverse_transform(Z_pca)

# Plot the first 10 images and their reconstructions
fig, ax = plt.subplots(2, 10, figsize=(10, 2))
for i in range(10):
    ax[0, i].imshow(X[i].view(28, 28).numpy(), cmap='gray')
    ax[0, i].axis('off')
    ax[1, i].imshow(X_reconstructed_pca[i].reshape(28, 28), cmap='gray')
    ax[1, i].axis('off')

In [ ]:
import pandas as pd

df = pd.DataFrame({"X": X[0], "X_reconstructed": X_reconstructed[0], "X_reconstructed_pca": X_reconstructed_pca[0]})

In [ ]:
df.plot()

In [ ]:
# Plot the original data and the projected data
plt.figure(figsize=(6, 6))
plt.scatter(X[:,0], X[:,1], label='Original data')
plt.scatter(Z[:,0], Z[:,0], label='Projected data')